# Phase 5: Full 80/20 Training & Validation Pipeline

This notebook implements the full training loop with an 80/20 train-validation split.


In [1]:
!pip install -q nibabel transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.7 MB/s eta 0:00:00:00:0100:01


In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
# ==================== IMPORTS AND SETUP ====================
import os
import glob
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import nibabel as nib
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
num_gpus = torch.cuda.device_count()
print(f"Number of GPUs available: {num_gpus} (only one will be used, batch size = 2)")

# ==================== PATHS ====================
DATA_DIR = "/kaggle/input/datasets/luumsk/asnr-miccai-brats-2023-gli-challenge-training-data/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"
MODEL_SAVE_PATH = "/kaggle/working/spiking_unet_best.pth"
CHECKPOINT_PATH = "/kaggle/working/spiking_unet_checkpoint.pth"
PATCH_SIZE = (64, 64, 64)
os.makedirs("/kaggle/working", exist_ok=True)

# ==================== DATASET CLASS (handles both structures) ====================
class BraTS3DDataset(Dataset):
    def __init__(self, patient_dirs, patch_size=PATCH_SIZE, val_mode=False):
        self.patient_dirs = patient_dirs
        self.patch_size = patch_size
        self.val_mode = val_mode

    def __len__(self):
        return len(self.patient_dirs)

    def __getitem__(self, idx):
        patient_dir = self.patient_dirs[idx]

        # ---- Segmentation (always a file directly in patient folder) ----
        seg_files = glob.glob(os.path.join(patient_dir, "*-seg.nii*"))
        if not seg_files:
            raise FileNotFoundError(f"No segmentation file in {patient_dir}")
        seg_path = seg_files[0]
        seg = nib.load(seg_path).get_fdata().astype(np.float32)

        # ---- Modalities: try flat file, then subfolder ----
        modalities = ['t1c', 't1n', 't2f', 't2w']
        images = []
        for mod in modalities:
            img_path = None
            # 1. Look for a direct file (not a directory)
            direct = glob.glob(os.path.join(patient_dir, f"*-{mod}.nii*"))
            direct_files = [f for f in direct if os.path.isfile(f)]
            if direct_files:
                img_path = direct_files[0]
            else:
                # 2. Look for a subdirectory
                subdirs = glob.glob(os.path.join(patient_dir, f"*-{mod}.nii*"))
                subdirs = [d for d in subdirs if os.path.isdir(d)]
                if subdirs:
                    inside = glob.glob(os.path.join(subdirs[0], "*.nii*"))
                    # Check that the file is not empty
                    non_empty = [f for f in inside if os.path.getsize(f) > 0]
                    if non_empty:
                        img_path = non_empty[0]
            if img_path is None:
                raise FileNotFoundError(f"Modality {mod} not found in {patient_dir}")
            img_data = nib.load(img_path).get_fdata().astype(np.float32)
            images.append(img_data)

        image_4d = np.stack(images, axis=0)   # (4, D, H, W)
        seg_3d = seg                           # (D, H, W)

        d, h, w = image_4d.shape[1:]
        pd, ph, pw = self.patch_size

        # Pad if needed
        if d < pd or h < ph or w < pw:
            pad_d = max(0, pd - d)
            pad_h = max(0, ph - h)
            pad_w = max(0, pw - w)
            image_4d = np.pad(image_4d, ((0,0), (0, pad_d), (0, pad_h), (0, pad_w)))
            seg_3d = np.pad(seg_3d, ((0, pad_d), (0, pad_h), (0, pad_w)))
            d, h, w = image_4d.shape[1:]

        # Patch extraction
        if self.val_mode:
            z, y, x = (d - pd)//2, (h - ph)//2, (w - pw)//2   # center
        else:
            z = np.random.randint(0, d - pd + 1)
            y = np.random.randint(0, h - ph + 1)
            x = np.random.randint(0, w - pw + 1)

        image_patch = image_4d[:, z:z+pd, y:y+ph, x:x+pw]
        seg_patch = seg_3d[z:z+pd, y:y+ph, x:x+pw]

        # Three mask channels
        mask_channels = np.stack([
            (seg_patch > 0),
            np.logical_or(seg_patch == 1, seg_patch == 3),
            (seg_patch == 3)
        ], axis=0).astype(np.float32)

        # Normalize each modality (foreground only)
        for c in range(4):
            img_c = image_patch[c]
            if img_c.max() > 0:
                fg = img_c > 0
                mean = img_c[fg].mean()
                std = img_c[fg].std() + 1e-8
                image_patch[c] = (img_c - mean) / std

        return {'image': torch.from_numpy(image_patch), 'mask': torch.from_numpy(mask_channels)}

# ==================== MODEL COMPONENTS ====================
class TernarySurrogate(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input, threshold=1.0):
        ctx.save_for_backward(input)
        ctx.threshold = threshold
        out = torch.zeros_like(input)
        out[input >= threshold] = 1.0
        out[input <= -threshold] = -1.0
        return out
    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        return grad_output * torch.exp(-torch.abs(input)), None

class TernaryLIF(nn.Module):
    def __init__(self, beta=0.9, threshold=1.0):
        super().__init__()
        self.beta, self.threshold = beta, threshold
        self.register_buffer("mem", None)
    def reset_state(self): self.mem = None
    def forward(self, x):
        if self.mem is None or self.mem.shape != x.shape:
            self.mem = torch.zeros_like(x)
        self.mem = self.beta * self.mem + x
        spk = TernarySurrogate.apply(self.mem, self.threshold)
        self.mem = self.mem - spk * self.threshold
        return spk

def shiftmax(x, dim=-1):
    x_norm = x - torch.max(x, dim=dim, keepdim=True)[0]
    approx_exp = 2.0 ** torch.floor(x_norm)
    return approx_exp / (torch.sum(approx_exp, dim=dim, keepdim=True) + 1e-8)

class BipolarSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.head_dim, self.num_heads = embed_dim // num_heads, num_heads
        self.q_proj, self.k_proj, self.v_proj, self.out_proj = [nn.Linear(embed_dim, embed_dim, bias=False) for _ in range(4)]
        self.lif_q, self.lif_k, self.lif_v = TernaryLIF(), TernaryLIF(), TernaryLIF()
    def reset_states(self): [l.reset_state() for l in [self.lif_q, self.lif_k, self.lif_v]]
    def forward(self, x):
        B, S, E = x.shape
        q_spk, k_spk, v_spk = self.lif_q(self.q_proj(x)), self.lif_k(self.k_proj(x)), self.lif_v(self.v_proj(x))
        q, k, v = [s.view(B, S, self.num_heads, self.head_dim).transpose(1, 2) for s in [q_spk, k_spk, v_spk]]
        attn = shiftmax(torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5), dim=-1)
        return self.out_proj(torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, S, E))

class SpikingTokenizer3D(nn.Module):
    def __init__(self, in_channels=4, embed_dim=32):
        super().__init__()
        self.conv1 = nn.Conv3d(in_channels, embed_dim//2, 3, 2, 1)
        self.conv2 = nn.Conv3d(embed_dim//2, embed_dim, 3, 2, 1)
        self.lif1 = TernaryLIF()
        self.lif2 = TernaryLIF()
    def reset_states(self): self.lif1.reset_state(); self.lif2.reset_state()
    def forward(self, x):
        s1 = self.lif1(self.conv1(x))
        return self.lif2(self.conv2(s1)), s1

class SpikingTransformerEncoder(nn.Module):
    def __init__(self, embed_dim, num_heads, depth):
        super().__init__()
        self.layers = nn.ModuleList([BipolarSelfAttention(embed_dim, num_heads) for _ in range(depth)])
    def reset_states(self): [l.reset_states() for l in self.layers]
    def forward(self, x):
        B, E, D, H, W = x.shape
        x_flat = x.view(B, E, D*H*W).transpose(1, 2)
        for layer in self.layers:
            x_flat = x_flat + layer(x_flat)
        return x_flat.transpose(1, 2).view(B, E, D, H, W)

class SpikingUNetDecoder3D(nn.Module):
    def __init__(self, embed_dim, out_channels=3):
        super().__init__()
        self.upconv1 = nn.ConvTranspose3d(embed_dim, embed_dim//2, 4, 2, 1)
        self.upconv2 = nn.ConvTranspose3d(embed_dim//2, out_channels, 4, 2, 1)
        self.lif_up1 = TernaryLIF()
    def reset_states(self): self.lif_up1.reset_state()
    def forward(self, encoder_spk, skip_spk):
        return self.upconv2(self.lif_up1(self.upconv1(encoder_spk) + skip_spk))

# ==================== AGENT AND FULL MODEL ====================
class Phi3TriageAgent:
    def __init__(self, load_model=True):
        self.load_model = load_model
        self.pipe = None
        if load_model: self._load_phi3()
    def _load_phi3(self):
        conf = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
        tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
        model = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3-mini-4k-instruct", quantization_config=conf, device_map="auto", trust_remote_code=False)
        self.pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
    def decide_routing(self, entropy, fg):
        if self.pipe is None: return True
        prompt = f"<|user|>Uncertainty: {entropy:.4f}. Foreground: {fg:.2f}. Route HIGH or LOW?<|end|><|assistant|>"
        out = self.pipe(prompt, max_new_tokens=5, temperature=0.1, do_sample=False)
        return "HIGH" in out[0]['generated_text'].upper()

class AgenticSpikingTransformerLLM(nn.Module):
    def __init__(self, agent_llm, in_channels=4, out_channels=3, embed_dim=32, num_heads=4, depth=2):
        super().__init__()
        self.tokenizer = SpikingTokenizer3D(in_channels, embed_dim)
        self.encoder = SpikingTransformerEncoder(embed_dim, num_heads, depth)
        self.decoder = SpikingUNetDecoder3D(embed_dim, out_channels)
        self.agent_llm = agent_llm
    def reset_all_states(self):
        self.tokenizer.reset_states()
        self.encoder.reset_states()
        self.decoder.reset_states()
    def forward_step(self, x):
        enc_spk, skip_spk = self.tokenizer(x)
        return self.decoder(self.encoder(enc_spk), skip_spk)
    def forward(self, x):
        self.reset_all_states()
        mem = self.forward_step(x)
        p = torch.sigmoid(mem)
        ent = (-p * torch.log(p+1e-6) - (1-p)*torch.log(1-p+1e-6)).mean().item()
        if self.agent_llm and self.agent_llm.decide_routing(ent, (p>0.5).sum().item()):
            for _ in range(3):
                mem += self.forward_step(x)
            return (mem / 4.0).unsqueeze(0)
        return mem.unsqueeze(0)

# ==================== METRICS AND VALIDATION ====================
def dice_coef(p, t, smooth=1e-5):
    p = (torch.sigmoid(p) > 0.5).float()
    intersection = (p * t).sum()
    union = p.sum() + t.sum()
    return (2. * intersection + smooth) / (union + smooth)

def validate(model, loader, criterion, use_routing=False):
    model.eval()
    total_loss, total_dice = 0, 0
    with torch.no_grad():
        for b in loader:
            model.reset_all_states()
            img, mask = b['image'].to(device), b['mask'].to(device)
            if use_routing:
                out = model(img).squeeze(0)
            else:
                out = model.forward_step(img)
            total_loss += criterion(out, mask).item()
            total_dice += dice_coef(out, mask).item()
            del img, mask, out
    return total_loss / len(loader), total_dice / len(loader)

# ==================== TRAINING LOOP ====================
def train_full(model, train_loader, val_loader, opt, crit, epochs=10, start_epoch=0, use_agent_in_train=False, resume_from=None):
    best_dice = 0
    scaler = torch.amp.GradScaler('cuda')

    if resume_from and os.path.exists(resume_from):
        checkpoint = torch.load(resume_from)
        state_dict = checkpoint['model_state_dict']
        if all(k.startswith('module.') for k in state_dict.keys()):
            state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
        model.load_state_dict(state_dict, strict=False)
        opt.load_state_dict(checkpoint['optimizer_state_dict'])
        if 'scaler_state_dict' in checkpoint:
            scaler.load_state_dict(checkpoint['scaler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_dice = checkpoint['best_dice']
        print(f"Resumed from epoch {start_epoch} with best Dice {best_dice:.4f}")

    for epoch in range(start_epoch, epochs):
        model.train()
        train_loss = 0
        for i, b in enumerate(train_loader):
            img, msk = b['image'].to(device), b['mask'].to(device)
            model.reset_all_states()
            opt.zero_grad()
            with torch.amp.autocast('cuda'):
                if use_agent_in_train:
                    out = model(img).squeeze(0)
                    loss = crit(out, msk)
                else:
                    l_sum = 0
                    for t in range(4):
                        out = model.forward_step(img)
                        l_sum += crit(out, msk)
                    loss = l_sum / 4.0
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            train_loss += loss.item()
            if i % 20 == 0:
                vram = torch.cuda.memory_reserved() / 1e9
                print(f"Epoch {epoch+1} | Batch {i} | Loss {loss.item():.4f} | VRAM: {vram:.2f}GB")
            del img, msk, out, loss
            gc.collect(); torch.cuda.empty_cache()

        v_loss, v_dice = validate(model, val_loader, crit, use_routing=use_agent_in_train)
        print(f"==> Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {v_loss:.4f} | Val Dice: {v_dice:.4f}")

        if v_dice > best_dice:
            best_dice = v_dice
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(f"Saved best model with Dice: {best_dice:.4f}")

        model.reset_all_states()
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': opt.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'best_dice': best_dice,
        }
        torch.save(checkpoint, CHECKPOINT_PATH)
        print(f"Saved checkpoint at epoch {epoch+1}")

# ==================== PREPARE DATA LOADERS ====================
raw_dirs = [os.path.join(DATA_DIR, d) for d in os.listdir(DATA_DIR) 
            if os.path.isdir(os.path.join(DATA_DIR, d))]
print(f"Raw patients: {len(raw_dirs)}")

def patient_has_all_modalities(patient_dir):
    # Check segmentation file exists and non-empty
    seg_files = glob.glob(os.path.join(patient_dir, "*-seg.nii*"))
    if not seg_files or os.path.getsize(seg_files[0]) == 0:
        return False
    for mod in ['t1c', 't1n', 't2f', 't2w']:
        found = False
        # Look for direct file
        direct = glob.glob(os.path.join(patient_dir, f"*-{mod}.nii*"))
        direct_files = [f for f in direct if os.path.isfile(f) and os.path.getsize(f) > 0]
        if direct_files:
            found = True
        else:
            # Look for subdirectory
            subdirs = glob.glob(os.path.join(patient_dir, f"*-{mod}.nii*"))
            subdirs = [d for d in subdirs if os.path.isdir(d)]
            for sub in subdirs:
                inside = glob.glob(os.path.join(sub, "*.nii*"))
                inside = [f for f in inside if os.path.getsize(f) > 0]
                if inside:
                    found = True
                    break
        if not found:
            return False
    return True

valid_dirs = [d for d in raw_dirs if patient_has_all_modalities(d)]
print(f"Valid patients: {len(valid_dirs)}")
if len(valid_dirs) == 0:
    raise RuntimeError("No valid patients found.")

train_dirs, val_dirs = train_test_split(valid_dirs, test_size=0.2, random_state=42)
print(f"Train: {len(train_dirs)}, Val: {len(val_dirs)}")

train_ds = BraTS3DDataset(train_dirs)
val_ds = BraTS3DDataset(val_dirs, val_mode=True)
train_ld = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=2)
val_ld = DataLoader(val_ds, batch_size=2, shuffle=False, num_workers=2)

# ==================== START FRESH TRAINING ====================
agent = Phi3TriageAgent(load_model=True)
model = AgenticSpikingTransformerLLM(agent).to(device)
print(f"Using device: {device}")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.BCEWithLogitsLoss()

# Optionally run a quick validation before training (model is randomly initialized, so Dice will be near 0)
v_loss, v_dice = validate(model, val_ld, criterion, use_routing=False)
print(f"Initial (random) validation Dice: {v_dice:.4f}")

# Start training from scratch (epoch 0)
print("Starting fresh training from epoch 1...")
train_full(model, train_ld, val_ld, optimizer, criterion,
           epochs=10, start_epoch=0, use_agent_in_train=True, resume_from=None)

Using device: cuda
Number of GPUs available: 2 (only one will be used, batch size = 2)
Raw patients: 1251


Valid patients: 1251
Train: 1000, Val: 251


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Epoch 10 | Batch 460 | Loss 0.0241 | VRAM: 14.29GB


Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_ne

Epoch 10 | Batch 480 | Loss 0.0070 | VRAM: 14.29GB


Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_ne

==> Epoch 10 | Train Loss: 0.0330 | Val Loss: 0.0771 | Val Dice: 0.7087
Saved checkpoint at epoch 10
